## Análisis exploratorio de las fuentes y del Datamart

Antes de modelar, se revisan las distribuciones, coberturas y valores faltantes de las variables principales. El objetivo es identificar patrones respaldados por los datos y evaluar qué variables son utilizables para el análisis posterior.

El datamart resume economía agropecuaria, incendios y características del Catastro Frutícola. DGA y ODEPA se describen por separado porque sus registros no tienen una correspondencia comunal disponible. Los precios de ODEPA se resumen por producto y unidad; no se comparan mezclando unidades distintas.

Estas tablas son descriptivas. Por sí solas no demuestran causalidad ni determinan qué amenaza tiene mayor efecto.

In [2]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

carpeta_datamart = Path("/content/drive/MyDrive/TRABAJO2 SI/datamart")

filas_esperadas = {
    "mart_comuna_anio": 200,
    "fact_incendios": 400,
    "dim_comuna": 10,
    "fact_caudal": 600,
    "dim_estacion": 5,
    "fact_precios": 267,
    "dim_producto": 138,
}

# Revisar primero todos los archivos para mostrar cualquier ausencia de una vez.
archivos_faltantes = [
    carpeta_datamart / f"{nombre}.csv"
    for nombre in filas_esperadas
    if not (carpeta_datamart / f"{nombre}.csv").is_file()
]
if archivos_faltantes:
    lista = "\n".join(f"- {ruta}" for ruta in archivos_faltantes)
    detalle_mart = (
        "\n`mart_comuna_anio.csv` se debe exportar desde el cuaderno que crea "
        "la tabla comuna-año; este cuaderno de EDA solo la carga."
        if any(ruta.name == "mart_comuna_anio.csv" for ruta in archivos_faltantes)
        else ""
    )
    raise FileNotFoundError(
        f"Faltan archivos requeridos en {carpeta_datamart}:\n{lista}{detalle_mart}"
    )

tablas_eda = {}
for nombre, filas in filas_esperadas.items():
    ruta_csv = carpeta_datamart / f"{nombre}.csv"
    tabla = pd.read_csv(ruta_csv, encoding="utf-8-sig")

    if len(tabla) != filas:
        raise ValueError(
            f"{nombre}: se esperaban {filas} filas y se encontraron {len(tabla)}."
        )

    tablas_eda[nombre] = tabla

mart_comuna_anio = tablas_eda["mart_comuna_anio"]
fact_incendios = tablas_eda["fact_incendios"]
dim_comuna = tablas_eda["dim_comuna"]
fact_caudal = tablas_eda["fact_caudal"]
dim_estacion = tablas_eda["dim_estacion"]
fact_precios = tablas_eda["fact_precios"]
dim_producto = tablas_eda["dim_producto"]

print("Tablas cargadas y verificadas:")
for nombre, tabla in tablas_eda.items():
    print(f"{nombre}: {tabla.shape}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TRABAJO2 SI/datamart/mart_comuna_anio.csv'

In [ ]:
# Resumen descriptivo del mart comuna-año
columnas_mart_eda = [
    "n_empresas_agro",
    "trab_dependientes_agro",
    "trab_ponderados_agro",
    "n_incendios",
    "sup_afectada_total",
    "pct_superficie_cerezo_2024",
    "pct_superficie_goteo_2024",
]

print("Resumen de indicadores del mart:")
display(
    mart_comuna_anio[columnas_mart_eda]
    .describe()
    .round(2)
    .T
)

# Incendios acumulados y datos faltantes por comuna
conaf_eda = fact_incendios.merge(
    dim_comuna[["id_comuna", "comuna"]],
    on="id_comuna",
    how="left",
    validate="many_to_one",
)

resumen_conaf = (
    conaf_eda
    .groupby("comuna")
    .agg(
        temporadas=("temporada", "nunique"),
        incendios_reportados=(
            "n_incendios",
            lambda s: s.sum(min_count=1),
        ),
        superficie_afectada_ha=(
            "sup_afectada_total",
            lambda s: s.sum(min_count=1),
        ),
        registros_sin_numero_incendios=(
            "n_incendios",
            lambda s: s.isna().sum(),
        ),
        registros_sin_superficie=(
            "sup_afectada_total",
            lambda s: s.isna().sum(),
        ),
    )
    .sort_values("superficie_afectada_ha", ascending=False)
)

print("\nCONAF: resumen por comuna:")
display(resumen_conaf.round(2))

# Cobertura y caudales por estación
dga_eda = fact_caudal.merge(
    dim_estacion[["id_estacion", "estacion"]],
    on="id_estacion",
    how="left",
    validate="many_to_one",
)

resumen_dga = (
    dga_eda
    .groupby("estacion")
    .agg(
        meses_registrados=("mes_num", "size"),
        mediciones_validas=("caudal_m3s", "count"),
        caudales_faltantes=("caudal_m3s", lambda s: s.isna().sum()),
        caudal_promedio_m3s=("caudal_m3s", "mean"),
        caudal_minimo_m3s=("caudal_m3s", "min"),
        caudal_maximo_m3s=("caudal_m3s", "max"),
    )
    .reset_index()
)

print("\nDGA: cobertura y resumen por estación:")
display(resumen_dga.round(2))

# ODEPA: cantidad de registros por producto y unidad
odepa_eda = fact_precios.merge(
    dim_producto[["id_producto", "producto", "unidad"]],
    on="id_producto",
    how="left",
    validate="many_to_one",
)

resumen_odepa = (
    odepa_eda
    .groupby(["producto", "unidad"], as_index=False)
    .size()
    .sort_values(["producto", "unidad"])
)

print(
    "\nODEPA: fechas:",
    odepa_eda["fecha"].min(),
    "a",
    odepa_eda["fecha"].max(),
)
print("ODEPA: registros incluidos:", len(odepa_eda))
print("\nODEPA: cantidad de registros por producto y unidad:")
display(resumen_odepa)

### Hallazgos descriptivos iniciales

En las 40 temporadas analizadas de CONAF, Pumanque registra la mayor superficie afectada acumulada (57.773,92 ha), mientras que San Fernando concentra el mayor número de incendios reportados (1.056). Son indicadores distintos: una comuna puede tener más eventos y otra acumular más superficie afectada. La superficie es la suma de lo reportado en distintas temporadas; no representa necesariamente hectáreas únicas.

En `mart_comuna_anio`, la superficie afectada por comuna y año presenta una distribución muy asimétrica: la mediana es 22,9 ha y el máximo, 46.490,08 ha. Por eso se analizará en un gráfico separado del número de incendios.

La cobertura de DGA varía según la estación. Puente Ruta 90 tiene 13 mediciones válidas de 120, por lo que su promedio debe interpretarse con cautela. ODEPA abarca cinco meses, de noviembre de 2024 a marzo de 2025; sus precios se analizarán por producto y unidad, sin mezclar unidades distintas.

Estos resultados son descriptivos. No demuestran que una amenaza haya causado un efecto económico.

## Visualización exploratoria de incendios y perfil comunal

Se grafican los incendios y la superficie afectada por año de término de temporada, además de los totales acumulados por comuna y los porcentajes estructurales del Catastro Frutícola 2024.

La superficie afectada se muestra en un gráfico separado debido a su alta variabilidad. Los porcentajes de cerezo y riego por goteo describen el catastro de 2024; no representan cambios históricos.

In [ ]:
import matplotlib.pyplot as plt

# Resumen anual CONAF, agrupado por el año de término de temporada.
conaf_anual = (
    fact_incendios
    .groupby("anio_fin", as_index=False)
    .agg(
        incendios=("n_incendios", lambda s: s.sum(min_count=1)),
        superficie_afectada_ha=(
            "sup_afectada_total",
            lambda s: s.sum(min_count=1),
        ),
    )
    .sort_values("anio_fin")
)

# Superficie afectada acumulada por comuna en las temporadas disponibles.
conaf_comunal = fact_incendios.merge(
    dim_comuna[["id_comuna", "comuna"]],
    on="id_comuna",
    how="left",
    validate="many_to_one",
)

superficie_comunal = (
    conaf_comunal
    .groupby("comuna")["sup_afectada_total"]
    .sum(min_count=1)
    .sort_values()
)

# Perfil estructural 2024: una fila por comuna.
perfil_2024 = mart_comuna_anio.loc[
    mart_comuna_anio["anio"].eq(2024),
    [
        "id_comuna",
        "pct_superficie_cerezo_2024",
        "pct_superficie_goteo_2024",
    ],
].merge(
    dim_comuna[["id_comuna", "comuna"]],
    on="id_comuna",
    how="left",
    validate="one_to_one",
).sort_values("pct_superficie_cerezo_2024")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(
    conaf_anual["anio_fin"],
    conaf_anual["incendios"],
    marker="o",
    linewidth=1.5,
)
axes[0, 0].set_title("Incendios reportados por año de término")
axes[0, 0].set_xlabel("Año")
axes[0, 0].set_ylabel("Número de incendios")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(
    conaf_anual["anio_fin"],
    conaf_anual["superficie_afectada_ha"],
    marker="o",
    linewidth=1.5,
    color="darkorange",
)
axes[0, 1].set_title("Superficie afectada por año de término")
axes[0, 1].set_xlabel("Año")
axes[0, 1].set_ylabel("Hectáreas")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].barh(
    superficie_comunal.index,
    superficie_comunal.values,
    color="firebrick",
)
axes[1, 0].set_title("Superficie afectada acumulada por comuna")
axes[1, 0].set_xlabel("Hectáreas acumuladas en las temporadas disponibles")
axes[1, 0].set_ylabel("Comuna")
axes[1, 0].grid(axis="x", alpha=0.3)

ancho_barra = 0.38
posiciones = range(len(perfil_2024))

axes[1, 1].bar(
    [p - ancho_barra / 2 for p in posiciones],
    perfil_2024["pct_superficie_cerezo_2024"],
    width=ancho_barra,
    label="Superficie de cerezo",
)
axes[1, 1].bar(
    [p + ancho_barra / 2 for p in posiciones],
    perfil_2024["pct_superficie_goteo_2024"],
    width=ancho_barra,
    label="Superficie con riego por goteo",
)
axes[1, 1].set_title("Perfil estructural del Catastro Frutícola 2024")
axes[1, 1].set_ylabel("Porcentaje de superficie")
axes[1, 1].set_xticks(list(posiciones))
axes[1, 1].set_xticklabels(perfil_2024["comuna"], rotation=45, ha="right")
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

# Confirmar que estén cargadas las tablas y resúmenes usados por el EDA.
tablas_esperadas = {
    "mart_comuna_anio": 200,
    "fact_incendios": 400,
    "fact_caudal": 600,
    "fact_precios": 267,
}

for nombre, filas_esperadas in tablas_esperadas.items():
    tabla = globals().get(nombre)

    if tabla is None:
        raise NameError(f"No está cargada la tabla {nombre}.")
    if len(tabla) != filas_esperadas:
        raise ValueError(
            f"{nombre}: se esperaban {filas_esperadas} filas; "
            f"se encontraron {len(tabla)}."
        )

# Comprobar los resúmenes del EDA.
resumenes_esperados = [
    "resumen_conaf",
    "resumen_dga",
    "resumen_odepa",
    "conaf_anual",
    "superficie_comunal",
    "perfil_2024",
]

for nombre in resumenes_esperados:
    if nombre not in globals():
        raise NameError(
            f"No está generado {nombre}. Ejecuta primero la celda de EDA."
        )

# Confirmar que el gráfico de cuatro paneles esté generado.
if "fig" not in globals() or len(fig.axes) != 4:
    raise ValueError("No encuentro el gráfico de cuatro paneles ejecutado.")

# Crear la carpeta donde quedará el notebook.
carpeta_analisis = (
    Path("/content/drive/MyDrive/TRABAJO2 SI")
    / "analisis de datos"
)
carpeta_analisis.mkdir(parents=True, exist_ok=True)

print("Validación del EDA: OK")
print("Tablas:", {nombre: len(globals()[nombre]) for nombre in tablas_esperadas})
print("Gráfico de cuatro paneles: OK")
print("Carpeta de destino lista:", carpeta_analisis)
print("Guarda el cuaderno desde Colab: Archivo > Guardar una copia en Drive.")